# Lab 7.2 &mdash; Build the Tracer

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 40 min &nbsp;|&nbsp; **Day 3 &middot; Module 7 &mdash; Multi-Agent System Evaluation**

### What you'll do
- Write the span store &mdash; and the one link that makes it a tree instead of a log
- Separate a span's own time from the time it spent inside its children
- Wire the store to LangChain as a real <code>BaseCallbackHandler</code>
- Rank by time and by tokens, and find they name different villains
- Send the tree to Langfuse and recognise every field

> **How this lab works.** You write real LangChain code &mdash; the agent under test, the callback
> handler that traces it, the typed verdict you grade. Fill every `BLANK`, then run the
> **Self-check** cell under each section. Those check the *objects you built* and the *recorded
> runs* shipped in the notebook, so they are deterministic and do not depend on the model.
> Cells marked **Run it for real** put your code in front of the sandbox model; that is the part
> worth watching, and it is never scored &mdash; scoring a live run would contradict Lab 7.1.

> **About sixty lines.** Once you have written a span tree by hand, a tracing product
> is a UI over something you understand rather than a black box you configure.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap, random, statistics
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-7-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Off is the default here because an eval lab makes a lot of calls.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 7 labs -- the same payment exceptions, now the
# subject of measurement rather than of engineering.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- carried forward from Module 1, Lab 1.2
# Real LangChain tools -- @tool turns a function into a tool object with a name, a schema
# and a description the model reads. Nothing to fill in; they are here so this notebook
# stands on its own and so the agent you measure is a real agent.

from langchain_core.tools import tool

@tool
def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1005'.

    Use when you need the status, amount, counterparty or reason code of a specific
    payment. Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


@tool
def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


EVAL_TOOLS = [lookup_payment, policy_for]
print("tools:", ", ".join(t.name for t in EVAL_TOOLS))

## Concept

A log is a list. A trace is a tree, and the tree is the information: it is the only thing that
lets you say the 3.9 seconds of retrieval *belonged to* the policy agent rather than merely
happening near it.

You build it in two halves, and the order matters. First the **store** &mdash; the spans and the links
between them, which is the idea. Then the **adapter** &mdash; the LangChain callback methods that feed
the store, which is just plumbing once you have the idea.

## Section 1 &mdash; The store, and what contains what

One dict per span. Each one knows when it started, when it ended, what it cost, and &mdash; the part
that matters &mdash; which span it began inside.

In [ ]:
class SpanStore:
    """Spans, and the links between them. No framework yet: this is the data structure."""

    def __init__(self):
        self.spans = {}          # span id -> span
        self.now = 0.0           # seconds since the run started

    def open_span(self, span_id, parent_id, name: str, kind: str) -> None:
        """Record a span that has just started."""
        self.spans[span_id] = {
            "id": span_id,
            "parent": parent_id,
            "name": name, "kind": kind,
            "t0": self.now, "t1": None, "tokens": 0, "status": "ok",
        }

    def close_span(self, span_id, tokens: int = 0, status: str = "ok") -> None:
        """Record a span that has just finished."""
        span = self.spans.get(span_id)
        if span is None:
            return               # an end with no start: nothing to close, and never a crash
        span["t1"] = self.now
        span["tokens"] = tokens
        span["status"] = status

    def ordered(self) -> list:
        """Spans in the order they started -- the order a trace viewer draws them."""
        return sorted(self.spans.values(), key=lambda s: (s["t0"], s["id"]))

In [ ]:
# ------------------------------------------------- one recorded run, event by event
# This is what LangChain's callbacks emitted during one multi-agent run: a start and an
# end per unit of work, each carrying its own id and the id of the span it began inside.
# The clock is recorded too, so every number in this lab is exact.
#
#   run
#     supervisor -> llm:supervisor
#     ledger     -> llm:ledger, lookup_payment
#     policy     -> llm:policy, retrieval
#     writer     -> llm:writer

RECORDED = [
    {"t": 0.0, "ev": "start", "id": "run",      "parent": None,     "name": "run",            "kind": "chain"},
    {"t": 0.0, "ev": "start", "id": "sup",      "parent": "run",    "name": "supervisor",     "kind": "chain"},
    {"t": 0.0, "ev": "start", "id": "sup.llm",  "parent": "sup",    "name": "llm:supervisor", "kind": "llm"},
    {"t": 0.4, "ev": "end",   "id": "sup.llm",  "tokens": 120},
    {"t": 0.4, "ev": "end",   "id": "sup"},
    {"t": 0.4, "ev": "start", "id": "led",      "parent": "run",    "name": "ledger",         "kind": "chain"},
    {"t": 0.4, "ev": "start", "id": "led.llm",  "parent": "led",    "name": "llm:ledger",     "kind": "llm"},
    {"t": 0.5, "ev": "end",   "id": "led.llm",  "tokens": 380},
    {"t": 0.5, "ev": "start", "id": "led.tool", "parent": "led",    "name": "lookup_payment", "kind": "tool"},
    {"t": 1.3, "ev": "end",   "id": "led.tool"},
    {"t": 1.3, "ev": "end",   "id": "led"},
    {"t": 1.3, "ev": "start", "id": "pol",      "parent": "run",    "name": "policy",         "kind": "chain"},
    {"t": 1.3, "ev": "start", "id": "pol.llm",  "parent": "pol",    "name": "llm:policy",     "kind": "llm"},
    {"t": 1.5, "ev": "end",   "id": "pol.llm",  "tokens": 420},
    {"t": 1.5, "ev": "start", "id": "pol.tool", "parent": "pol",    "name": "retrieval",      "kind": "tool"},
    {"t": 5.4, "ev": "end",   "id": "pol.tool"},
    {"t": 5.4, "ev": "end",   "id": "pol"},
    {"t": 5.4, "ev": "start", "id": "wri",      "parent": "run",    "name": "writer",         "kind": "chain"},
    {"t": 5.4, "ev": "start", "id": "wri.llm",  "parent": "wri",    "name": "llm:writer",     "kind": "llm"},
    {"t": 7.1, "ev": "end",   "id": "wri.llm",  "tokens": 2260},
    {"t": 7.1, "ev": "end",   "id": "wri"},
    {"t": 7.1, "ev": "end",   "id": "run"},
]

print(f"{len(RECORDED)} recorded events, "
      f"{sum(1 for e in RECORDED if e['ev'] == 'start')} spans")

In [ ]:
def replay(events: list) -> SpanStore:
    """Feed recorded events to a fresh store, driving its clock by hand."""
    store = SpanStore()
    for e in events:
        store.now = e["t"]
        if e["ev"] == "start":
            store.open_span(e["id"], e["parent"], e["name"], e["kind"])
        else:
            store.close_span(e["id"], e.get("tokens", 0))
    return store


def spans() -> list:
    """The recorded run, as a list of spans."""
    return replay(RECORDED).ordered()


def by_name(name: str) -> dict:
    return next(s for s in spans() if s["name"] == name)


def children(all_spans: list, span_id) -> list:
    return [s for s in all_spans if s["parent"] == span_id]

In [ ]:
# --- Self-check: Section 1   (a replayed span tree -- no model call)
check("every span was closed",
      lambda: all(s["t1"] is not None for s in spans()))
check("the root has no parent",
      lambda: by_name("run")["parent"] is None)
check("the supervisor's parent is the run",
      lambda: by_name("supervisor")["parent"] == by_name("run")["id"])
check("RETRIEVAL'S PARENT IS THE POLICY AGENT, not the run",
      lambda: by_name("retrieval")["parent"] == by_name("policy")["id"],
      "this one link is the difference between a trace and a log")
check("the tool call sits inside the ledger agent",
      lambda: by_name("lookup_payment")["parent"] == by_name("ledger")["id"])
check("every span except the root has a parent that exists",
      lambda: all(s["parent"] in {x["id"] for x in spans()}
                  for s in spans() if s["parent"] is not None),
      "a dangling parent id is how a trace viewer silently drops half a run")
check("the whole run took 7.1 seconds",
      lambda: abs(by_name("run")["t1"] - by_name("run")["t0"] - 7.1) < 1e-9)
check("the run has four children, one per agent",
      lambda: len(children(spans(), by_name("run")["id"])) == 4)

## Section 2 &mdash; Its own time, and its children's

The policy agent took 4.1 seconds. It *spent* 0.2 of them. Attribution needs the difference, and
the difference only exists because you kept the tree.

In [ ]:
def total_time(all_spans: list, span_id) -> float:
    """Wall clock from the moment this span opened to the moment it closed.

    A span that never closed -- a timeout, a crash -- counts as zero here rather than
    taking the whole report down with it.
    """
    s = next(x for x in all_spans if x["id"] == span_id)
    return round(s["t1"] - s["t0"], 6) if s["t1"] is not None else 0.0


def self_time(all_spans: list, span_id) -> float:
    """Time inside this span that was NOT spent inside one of its children."""
    own = total_time(all_spans, span_id)
    in_children = sum(total_time(all_spans, c["id"]) for c in children(all_spans, span_id))
    return round(own - in_children, 6)


def tokens_including_children(all_spans: list, span_id) -> int:
    """What this span cost, counting everything that ran inside it."""
    s = next(x for x in all_spans if x["id"] == span_id)
    return s["tokens"] + sum(tokens_including_children(all_spans, c["id"])
                             for c in children(all_spans, span_id))

In [ ]:
# --- Self-check: Section 2
def sp():
    return spans()

check("the policy agent's total is 4.1 seconds",
      lambda: abs(total_time(sp(), by_name("policy")["id"]) - 4.1) < 1e-9)
check("but it spent none of them itself",
      lambda: abs(self_time(sp(), by_name("policy")["id"])) < 1e-9,
      "the policy agent is not slow -- it contains something slow, and only the tree says so")
check("its own model call took 0.2s",
      lambda: abs(self_time(sp(), by_name("llm:policy")["id"]) - 0.2) < 1e-9)
check("retrieval has no children, so its self time is its total",
      lambda: self_time(sp(), by_name("retrieval")["id"])
              == total_time(sp(), by_name("retrieval")["id"]))
check("the self times of every span add up to the whole run",
      lambda: abs(sum(self_time(sp(), s["id"]) for s in sp())
                  - total_time(sp(), by_name("run")["id"])) < 1e-9,
      "if this does not hold, the tree is wrong and every attribution built on it is wrong")
check("the run's token total includes everything beneath it",
      lambda: tokens_including_children(sp(), by_name("run")["id"]) == 3180)
check("the policy agent is charged for the model call inside it",
      lambda: tokens_including_children(sp(), by_name("policy")["id"]) == 420)
check("a tool span costs no tokens of its own",
      lambda: by_name("retrieval")["tokens"] == 0,
      "retrieval is 55% of the wall clock and 0% of the bill -- hold on to that")

## Section 3 &mdash; The same store, driven by LangChain

Now the plumbing. `BaseCallbackHandler` is the interface LangChain calls as a run proceeds: it
hands you a `run_id` for the unit of work and a `parent_run_id` for whatever it started inside.

Every method below is a two-line adapter onto the store you already wrote.

In [ ]:
def tokens_from(response) -> int:
    """Total tokens off an LLMResult, from whichever place this provider put them."""
    usage = (getattr(response, "llm_output", None) or {}).get("token_usage") or {}
    if usage.get("total_tokens"):
        return int(usage["total_tokens"])
    for batch in getattr(response, "generations", None) or []:
        for gen in batch:
            meta = getattr(getattr(gen, "message", None), "usage_metadata", None) or {}
            if meta.get("total_tokens"):
                return int(meta["total_tokens"])
    return 0

In [ ]:
from langchain_core.callbacks import BaseCallbackHandler

class SpanTracer(BaseCallbackHandler):
    """A real LangChain tracer. Pass it as config={"callbacks": [tracer]} to any invoke()."""

    def __init__(self):
        self.store = SpanStore()
        self.t0 = time.perf_counter()

    def _start(self, run_id, parent_run_id, name, kind):
        self.store.now = round(time.perf_counter() - self.t0, 3)
        self.store.open_span(str(run_id), str(parent_run_id) if parent_run_id else None,
                             name, kind)

    def _end(self, run_id, tokens=0, status="ok"):
        self.store.now = round(time.perf_counter() - self.t0, 3)
        self.store.close_span(str(run_id), tokens, status)

    # --- LangChain calls these. A chain is an agent or a graph node. ---
    def on_chain_start(self, serialized, inputs, *, run_id=None, parent_run_id=None, **kw):
        name = kw.get("name") or (serialized or {}).get("name") or "chain"
        self._start(run_id, parent_run_id, name, "chain")

    def on_chain_end(self, outputs, *, run_id=None, **kw):
        self._end(run_id)

    def on_chain_error(self, error, *, run_id=None, **kw):
        self._end(run_id, status="error")

    # --- a chat model gets on_chat_model_start; a completion model gets on_llm_start ---
    def on_chat_model_start(self, serialized, messages, *, run_id=None, parent_run_id=None, **kw):
        self._start(run_id, parent_run_id, "llm", "llm")

    def on_llm_start(self, serialized, prompts, *, run_id=None, parent_run_id=None, **kw):
        self._start(run_id, parent_run_id, "llm", "llm")

    def on_llm_end(self, response, *, run_id=None, **kw):
        self._end(run_id, tokens=tokens_from(response))

    def on_llm_error(self, error, *, run_id=None, **kw):
        self._end(run_id, status="error")

    # --- tools ---
    def on_tool_start(self, serialized, input_str, *, run_id=None, parent_run_id=None, **kw):
        self._start(run_id, parent_run_id, (serialized or {}).get("name") or "tool", "tool")

    def on_tool_end(self, output, *, run_id=None, **kw):
        self._end(run_id)

    def on_tool_error(self, error, *, run_id=None, **kw):
        self._end(run_id, status="error")

In [ ]:
# --- Self-check: Section 3   (the handler driven by hand -- no model call)
def hand_driven() -> SpanStore:
    """Call the callback methods the way LangChain would, with ids of our own."""
    t = SpanTracer()
    t.on_chain_start({"name": "ledger"}, {}, run_id="r1", parent_run_id=None)
    t.on_tool_start({"name": "lookup_payment"}, "PMT-1005", run_id="r2", parent_run_id="r1")
    t.on_tool_end("{}", run_id="r2")
    t.on_tool_start({"name": "release_payment"}, "PMT-1005", run_id="r3", parent_run_id="r1")
    t.on_tool_error(RuntimeError("approval gate refused"), run_id="r3")
    t.on_chain_end({}, run_id="r1")
    return t.store

check("the tracer is a LangChain callback handler",
      lambda: isinstance(SpanTracer(), BaseCallbackHandler),
      "which is the only reason invoke(config={'callbacks': [...]}) will accept it")
check("it records one span per unit of work",
      lambda: len(hand_driven().spans) == 3)
check("a tool started inside a chain becomes that chain's child",
      lambda: hand_driven().spans["r2"]["parent"] == "r1",
      "same link as Section 1 -- LangChain hands you the parent id, you just have to keep it")
check("the outermost span has no parent",
      lambda: hand_driven().spans["r1"]["parent"] is None)
check("a tool that raised is recorded as an error, not lost",
      lambda: hand_driven().spans["r3"]["status"] == "error",
      "Lab 7.4 diagnoses failures off exactly this field")
check("an end with no start does not crash the tracer",
      lambda: SpanTracer().store.close_span("never-started") is None,
      "callbacks arrive out of order more often than you would like")

## Section 4 &mdash; Two different villains

Roll the tree up and rank it twice: once by time, once by tokens. The tree gives you two numbers
per span, and choosing between them is the whole of attribution.

In [ ]:
def breakdown(all_spans: list) -> list:
    """One row per span, root excluded -- including the root would double-count everything."""
    return [{"name": s["name"], "kind": s["kind"],
             "self_s": self_time(all_spans, s["id"]),
             "total_s": total_time(all_spans, s["id"]),
             "tokens": s["tokens"]}
            for s in all_spans if s["parent"] is not None]


def slowest_step(all_spans: list) -> str:
    """Which span do you go and optimise?"""
    key = "self_s"
    return max(breakdown(all_spans), key=lambda r: r[key])["name"]


def dearest_step(all_spans: list) -> str:
    """Which span do you go and make cheaper?"""
    return max(breakdown(all_spans), key=lambda r: r["tokens"])["name"]


def share(all_spans: list, name: str, key: str) -> float:
    rows = breakdown(all_spans)
    total = sum(r[key] for r in rows)
    row = next(r for r in rows if r["name"] == name)
    return row[key] / total if total else 0.0

In [ ]:
# --- Self-check: Section 4
check("the slowest step is retrieval",
      lambda: slowest_step(sp()) == "retrieval")
check("RANKING BY total_s WOULD HAVE NAMED THE POLICY AGENT INSTEAD",
      lambda: max(breakdown(sp()), key=lambda r: r["total_s"])["name"] == "policy",
      "the agent that contains the slow step, whose own code you could rewrite all week "
      "without moving the number")
check("the dearest step is the writer's model call",
      lambda: dearest_step(sp()) == "llm:writer")
check("THE SLOWEST AND THE DEAREST ARE NOT THE SAME STEP",
      lambda: slowest_step(sp()) != dearest_step(sp()),
      "optimise the biggest number on the wrong axis and you work hard and save nothing")
check("retrieval is more than half the wall clock",
      lambda: share(sp(), "retrieval", "self_s") > 0.5)
check("and none of the bill",
      lambda: share(sp(), "retrieval", "tokens") == 0.0)
check("the writer is most of the bill",
      lambda: share(sp(), "llm:writer", "tokens") > 0.7)
check("the root is excluded from the breakdown, or everything double-counts",
      lambda: all(r["name"] != "run" for r in breakdown(sp())))

def _report():
    rows = sorted(breakdown(spans()), key=lambda r: -r["self_s"])
    print(f"  {'span':18}{'kind':8}{'self s':>9}{'total s':>9}{'tokens':>9}")
    print("  " + "-" * 54)
    for r in rows:
        print(f"  {r['name']:18}{r['kind']:8}{r['self_s']:>9.1f}{r['total_s']:>9.1f}"
              f"{r['tokens']:>9}")
    print()
    print(f"  slowest step : {slowest_step(spans())}  "
          f"({share(spans(), slowest_step(spans()), 'self_s'):.0%} of the wall clock)")
    print(f"  dearest step : {dearest_step(spans())}  "
          f"({share(spans(), dearest_step(spans()), 'tokens'):.0%} of the bill)")
guard(_report)

## Section 5 &mdash; What a tracing product calls each of these

Every field you built has a name in a tracing product: your span is a *span*, a model call is a
*generation*, the whole thing is a *trace*, and `parent` is what draws the tree.

The distinction between a span and a generation is not cosmetic. Langfuse fills its token and
cost columns from **generations only** &mdash; send a model call as a plain span and the project
reports zero tokens however much it actually spent.

In [ ]:
def observation_type(span: dict) -> str:
    """What a tracing product should call this span: 'generation' or 'span'.

    An agent that contains a model call is not itself a model call; sending it as one
    double-counts the tokens under the parent as well as the child.
    """
    return "generation" if span["kind"] == "llm" else "span"

In [ ]:
# --- Self-check: Section 5   (a pure mapping -- no Langfuse, no network, no model)
check("a model call is a generation",
      lambda: observation_type(by_name("llm:writer")) == "generation")
check("a tool call is a plain span",
      lambda: observation_type(by_name("retrieval")) == "span")
check("AN AGENT THAT CONTAINS A MODEL CALL IS NOT ITSELF ONE",
      lambda: observation_type(by_name("policy")) == "span",
      "sending the parent as a generation too reports 420 tokens twice")
check("every span that recorded tokens is a generation",
      lambda: all(observation_type(s) == "generation" for s in sp() if s["tokens"] > 0),
      "the ones that are not are exactly the ones whose cost the product will never show")
check("exactly four of the eleven spans are generations",
      lambda: sum(1 for s in sp() if observation_type(s) == "generation") == 4)

## Run it for real &mdash; trace an actual agent

Same tracer, no replay. Attach it to a real `create_agent` run and look at what LangChain
actually emits &mdash; you will see more chain spans than the recorded run had, because LangGraph
emits one per node.

In [ ]:
if llm_ready():
    from langchain.agents import create_agent

    def _trace_a_real_run():
        tracer = SpanTracer()
        agent = create_agent(model=get_llm(), tools=EVAL_TOOLS,
                             system_prompt="You are a payments operations analyst. "
                                           "Look the payment up, read its policy, then answer.")
        agent.invoke({"messages": [("human", "What should we do about PMT-1005?")]},
                     config={"callbacks": [tracer]})

        live = tracer.store.ordered()
        print(f"  {len(live)} spans recorded, "
              f"{sum(1 for s in live if s['kind'] == 'llm')} of them model calls")
        for r in sorted(breakdown(live), key=lambda r: -r["self_s"])[:8]:
            print(f"    {r['name'][:26]:28}{r['kind']:7}{r['self_s']:>8.2f}s"
                  f"{r['tokens']:>8} tok")
        print(f"\n  slowest: {slowest_step(live)}    dearest: {dearest_step(live)}")
    guard(_trace_a_real_run)

## Run it for real &mdash; send the tree to Langfuse

This cell sends the **recorded** run to Langfuse if it is configured. It reads its settings from
the environment and hardcodes nothing &mdash; in particular no host, because Langfuse keys are
**region-bound**: a key pair issued in one region is rejected by another.

Nothing above needed Langfuse, and nothing you were scored on touches it.

In [ ]:
def langfuse_settings():
    """Base URL, public key, secret key -- from the environment, never hardcoded."""
    host = os.environ.get("LANGFUSE_BASE_URL") or os.environ.get("LANGFUSE_HOST")
    return host, os.environ.get("LANGFUSE_PUBLIC_KEY"), os.environ.get("LANGFUSE_SECRET_KEY")


def send_to_langfuse():
    host, pk, sk = langfuse_settings()
    if not (host and pk and sk):
        print("Langfuse is not configured in this sandbox. To point at one, set:")
        print("  export LANGFUSE_BASE_URL=...      # the base URL for YOUR region")
        print("  export LANGFUSE_PUBLIC_KEY=pk-lf-...")
        print("  export LANGFUSE_SECRET_KEY=sk-lf-...")
        print()
        print("Nothing above needed it. The span tree you built carries the same information,")
        print("and the mapping is one line per field:")
        for ours, theirs in (("run span", "trace"), ("chain span", "span"),
                             ("llm span", "generation"), ("parent", "the tree itself"),
                             ("tokens", "usage_details"), ("your assertions", "scores")):
            print(f"    {ours:16} -> {theirs}")
        return

    from langfuse import Langfuse
    lf = Langfuse(host=host, public_key=pk, secret_key=sk)
    if not lf.auth_check():
        print("Langfuse credentials rejected. Keys are region-bound -- check that this key "
              "pair belongs to the host in LANGFUSE_BASE_URL.")
        return

    all_spans = spans()

    def emit(span):
        # Observations nest by being entered inside one another -- that IS your parent link.
        with lf.start_as_current_observation(name=span["name"],
                                             as_type=observation_type(span)) as obs:
            obs.update(metadata={"kind": span["kind"],
                                 "self_s": self_time(all_spans, span["id"]),
                                 "total_s": total_time(all_spans, span["id"])})
            if observation_type(span) == "generation":
                # usage_details is the ONLY field Langfuse fills its token columns from.
                obs.update(model=LLM_MODEL or "unknown",
                           usage_details={"input": 0, "output": span["tokens"]})
            for child in children(all_spans, span["id"]):
                emit(child)

    emit(next(s for s in all_spans if s["parent"] is None))
    lf.flush()
    print(f"sent one trace to {host}")
    print("Open it and compare the tree with the one you printed above.")

guard(send_to_langfuse)

## Run it for real &mdash; your own tokens

The trace above is the recorded run, replayed. This cell makes **one real model call** and lets
Langfuse instrument it for you: `langfuse.openai` is a drop-in for the `openai` client that emits
the generation, the model name and the true token counts with no other change to your code.

In [ ]:
def trace_a_real_call():
    host, pk, sk = langfuse_settings()
    if not (host and pk and sk and llm_ready()):
        print("Langfuse or the model is not configured here -- see the previous cell.")
        return

    from langfuse import Langfuse, get_client
    Langfuse(host=host, public_key=pk, secret_key=sk)     # configures the client this run uses

    # The ONLY change from a normal call is the import. Everything else is the openai SDK.
    from langfuse.openai import openai
    client = openai.OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)
    r = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[{"role": "user",
                   "content": "In one sentence: why does a payment exception need a human?"}],
        extra_body=NO_THINK,
        name="lab-7-2-real-call",          # what the trace is called in the UI
    )
    get_client().flush()                   # the SDK batches; a notebook can exit before it sends

    print((r.choices[0].message.content or "").strip())
    print(f"\n{r.usage.prompt_tokens} in / {r.usage.completion_tokens} out")
    print("Those two numbers are now on the generation in Langfuse. Read cohort TOTALS off the")
    print("tokenomics dashboard, not off Langfuse -- this model has no priced entry there, so")
    print("its cost column stays empty however many tokens the generation carries.")

guard(trace_a_real_call)

### Read it

If Langfuse is not wired up in your sandbox, you have lost nothing today: the mapping printed
above is the whole of what a tracing product adds on top of what you just built, plus storage, a
UI and a place to attach scores.

That is worth having in production and it is not worth being mystified by. The thing to take away
is the shape &mdash; **spans with parents, timing, usage, and scores attached to a trace id** &mdash; because
every vendor implements that shape and you can now read any of them.

In [ ]:
score()

## Your turn

1. `on_chain_start` names a span from whatever LangChain passed. Run the live cell again and look
   at the names: some are useful, some are not. Add `metadata={"agent": "policy"}` to one
   `invoke` and read it back off `kw` in the handler.
2. The store never notices an unclosed span. Add a check that reports any span with `t1 is None`
   at the end of a run &mdash; that is what a timeout looks like from inside a tracer.
3. `share(..., "self_s")` treats every second as equal. A second of retrieval and a second of
   model latency have different owners and different fixes. Split the breakdown by `kind` and see
   whether the priority changes.